# Experiments

This file contains multiple experiments that were done in order to find the solution to the [task](../task.md). Most of the necesary code is located in [solution.py](solution.py).

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import solution
from torch.utils.data import Subset, DataLoader
from sklearn.model_selection import KFold

from solution import read_trainset, NetConfig, NeuralNetwork

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"Device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 4060 Laptop GPU


## Method of evaluation
In order to make sure that the following experiments are reliable we used cross-validation for measuring prediction quality. The accuracy itself is measured just like in the [helpers.py](../helpers.py) file provided.

In [2]:
def calc_macro_accuracy(predictions: np.ndarray, targets: np.ndarray, n_classes: int = 50):
    assert len(predictions) == len(targets)
    accuracies = []
    for class_idx in range(n_classes):
        mask = targets == class_idx
        n_samples = mask.sum()
        if n_samples > 0:
            accuracies.append((predictions[mask] == class_idx).sum() / n_samples)
        else:
            accuracies.append(0.0)
    return float(np.mean(accuracies))


def make_dataloader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=4,
        pin_memory=torch.cuda.is_available(),
        prefetch_factor=2,
        persistent_workers=True,
    )


def evaluate_cross_validation(experiment_name, model_instance, full_dataset, number_of_folds: int = 3):
    kfold = KFold(n_splits=number_of_folds, shuffle=True, random_state=42)
    dataset_indices = np.arange(len(full_dataset))
    accuracy_scores = []

    initial_state = copy.deepcopy(model_instance.model.state_dict())

    for fold_number, (train_indices, validation_indices) in enumerate(kfold.split(dataset_indices), start=1):
        model_copy = copy.deepcopy(model_instance)
        model_copy.model.load_state_dict(initial_state)

        train_subset = Subset(full_dataset, train_indices)
        validation_subset = Subset(full_dataset, validation_indices)

        train_dataloader = make_dataloader(train_subset, batch_size=model_copy.config.batch_size, shuffle=True)
        validation_dataloader = make_dataloader(validation_subset, batch_size=model_copy.config.batch_size, shuffle=False)

        model_copy.fit(train_dataloader, print_epochs=False)
        predictions = np.array([prediction for _, prediction in model_copy.predict(validation_dataloader)])
        true_labels = np.array([full_dataset.targets[i] for i in validation_indices])

        fold_accuracy = calc_macro_accuracy(predictions, true_labels, n_classes=NUM_CLASSES)
        accuracy_scores.append(fold_accuracy)

    mean_accuracy = float(np.mean(accuracy_scores))
    std_accuracy = float(np.std(accuracy_scores))

    return {
        "experiment": experiment_name,
        "mean_macro_accuracy": mean_accuracy,
        "std_macro_accuracy": std_accuracy,
        "folds": number_of_folds,
        "epochs": model_instance.config.epochs,
        "batch_size": model_instance.config.batch_size,
        "learning_rate": model_instance.config.learning_rate,
        "batch_norm": model_instance.config.use_batch_normalization,
        "conv_layers": str(model_instance.config.convolutional_layers),
        "fc_layers": str(model_instance.config.fully_connected_layers),
        "dropout": str(model_instance.config.dropout_rates),
        "activation": str(model_instance.config.activation_types),
    }


def compare_models(experiments, training_data, number_of_folds: int = 3, model_factory=None):
    results = []
    for experiment_name, config in experiments:
        model = model_factory(config) if model_factory is not None else NeuralNetwork(config, num_classes=NUM_CLASSES)
        result = evaluate_cross_validation(
            experiment_name=experiment_name,
            model_instance=model,
            full_dataset=training_data,
            number_of_folds=number_of_folds,
        )
        results.append(result)

    results_df = pd.DataFrame(results).sort_values(by="mean_macro_accuracy", ascending=False).reset_index(drop=True)
    return results_df

## Experiments

Reading in data with different IMG sizes and different augmentation configurations.

In [3]:
_, train_data_default = read_trainset(augmentation="none", img_size=64)
_, train_data_weak = read_trainset(augmentation="weak", img_size=64)
_, train_data_strong = read_trainset(augmentation="strong", img_size=64)

_, train_data_96 = read_trainset(augmentation="weak", img_size=96)
_, train_data_128 = read_trainset(augmentation="weak", img_size=128)

NUM_CLASSES = len(train_data_default.classes)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Train size: {len(train_data_default)}")

Number of classes: 42
Train size: 74490


### Image size


In [4]:
def make_model_for_img_size(img_size: int):
    def _factory(config: NetConfig):
        old_img_size = solution.IMG_SIZE
        solution.IMG_SIZE = img_size
        try:
            return NeuralNetwork(config, num_classes=NUM_CLASSES)
        finally:
            solution.IMG_SIZE = old_img_size
    return _factory

In [ ]:
image_size_experiments = [
    ("IMG_SIZE = 64", 64, NetConfig(epochs=15, batch_size=64, learning_rate=3e-4)),
    ("IMG_SIZE = 96", 96, NetConfig(epochs=15, batch_size=64, learning_rate=3e-4)),
    ("IMG_SIZE = 128", 128, NetConfig(epochs=15, batch_size=64, learning_rate=3e-4)),
]

image_size_result_frames = []
for experiment_name, img_size, config in image_size_experiments:
    if img_size == 64:    dataset = train_data_weak
    elif img_size == 96:  dataset = train_data_96
    elif img_size == 128: dataset = train_data_128
    else:                 raise ValueError(f"Unsupported image size: {img_size}")

    result_df = compare_models( [(experiment_name, config)], training_data=dataset,  model_factory=make_model_for_img_size(img_size), )
    result_df["img_size"] = img_size
    image_size_result_frames.append(result_df)

pd.concat(image_size_result_frames, ignore_index=True).sort_values(by="mean_macro_accuracy", ascending=False).reset_index(drop=True)

,experiment,mean_macro_accuracy,std_macro_accuracy,folds,epochs,batch_size,learning_rate,batch_norm,conv_layers,fc_layers,dropout,activation,img_size
0,IMG_SIZE = 96,0.591555,0.007626,3,15,64,0.0003,True,"[32, 64, 128, 256]","[512, 128]",[0.2],[<class 'torch.nn.modules.activation.ReLU'>],96
1,IMG_SIZE = 64,0.587695,0.007918,3,15,64,0.0003,True,"[32, 64, 128, 256]","[512, 128]",[0.2],[<class 'torch.nn.modules.activation.ReLU'>],64
2,IMG_SIZE = 128,0.582165,0.002664,3,15,64,0.0003,True,"[32, 64, 128, 256]","[512, 128]",[0.2],[<class 'torch.nn.modules.activation.ReLU'>],128


### Augmentation


In [6]:
augmentation_experiments = [
    ("No augmentation", NetConfig(epochs=15, batch_size=64, learning_rate=3e-4)),
    ("Weak augmentation", NetConfig(epochs=15, batch_size=64, learning_rate=3e-4)),
    ("Strong augmentation", NetConfig(epochs=15, batch_size=64, learning_rate=3e-4)),
]

pd.concat([
    compare_models([augmentation_experiments[0]], training_data=train_data_default),
    compare_models([augmentation_experiments[1]], training_data=train_data_weak),
    compare_models([augmentation_experiments[2]], training_data=train_data_strong),
], ignore_index=True).sort_values(by="mean_macro_accuracy", ascending=False).reset_index(drop=True)


,experiment,mean_macro_accuracy,std_macro_accuracy,folds,epochs,batch_size,learning_rate,batch_norm,conv_layers,fc_layers,dropout,activation
0,Weak augmentation,0.587952,0.003989,3,15,64,0.0003,True,"[32, 64, 128, 256]","[512, 128]",[0.2],[<class 'torch.nn.modules.activation.ReLU'>]
1,Strong augmentation,0.573681,0.005238,3,15,64,0.0003,True,"[32, 64, 128, 256]","[512, 128]",[0.2],[<class 'torch.nn.modules.activation.ReLU'>]
2,No augmentation,0.533410,0.003986,3,15,64,0.0003,True,"[32, 64, 128, 256]","[512, 128]",[0.2],[<class 'torch.nn.modules.activation.ReLU'>]


### Batch Normalization & Batch sizes

In [7]:
bn_batch_experiments = [
    ("No BN + Batch=32",  NetConfig(use_batch_normalization=False, batch_size=32,  epochs=15, learning_rate=3e-4)),
    ("No BN + Batch=64",  NetConfig(use_batch_normalization=False, batch_size=64,  epochs=15, learning_rate=3e-4)),
    ("No BN + Batch=128", NetConfig(use_batch_normalization=False, batch_size=128, epochs=15, learning_rate=3e-4)),
    ("BN + Batch=32",     NetConfig(use_batch_normalization=True,  batch_size=32,  epochs=15, learning_rate=3e-4)),
    ("BN + Batch=64",     NetConfig(use_batch_normalization=True,  batch_size=64,  epochs=15, learning_rate=3e-4)),
    ("BN + Batch=128",    NetConfig(use_batch_normalization=True,  batch_size=128, epochs=15, learning_rate=3e-4)),
]

compare_models(bn_batch_experiments, training_data=train_data_weak)

PermissionError: Caught PermissionError in DataLoader worker process 2.
Original Traceback (most recent call last):
  File "c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\_utils\worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 50, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataset.py", line 420, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataset.py", line 420,

### Net layers and depth

In [ ]:

layer_experiments = [
    (
        "Tiny",
        NetConfig(
            convolutional_layers=[32],
            fully_connected_layers=[128],
            dropout_rates=[0.2],
            epochs=15,
            batch_size=64,
            learning_rate=3e-4,
        )
    ),
    (
        "Shallow",
        NetConfig(
            convolutional_layers=[32, 64],
            fully_connected_layers=[256],
            dropout_rates=[0.2],
            epochs=15,
            batch_size=64,
            learning_rate=3e-4,
        )
    ),
    (
        "Medium",
        NetConfig(
            convolutional_layers=[32, 64, 128],
            fully_connected_layers=[512, 128],
            dropout_rates=[0.2],
            epochs=15,
            batch_size=64,
            learning_rate=3e-4,
        )
    ),
    (
        "Deep",
        NetConfig(
            convolutional_layers=[32, 64, 128, 256],
            fully_connected_layers=[512, 256, 128],
            dropout_rates=[0.2],
            epochs=15,
            batch_size=64,
            learning_rate=3e-4,
        )
    ),
]

compare_models(layer_experiments, training_data=train_data_weak)

### Optimizer + Weight Decay

In [ ]:
# ADAM, SGD (JAKIE MOMENTUM?), ADAMW

### Learning rate + scheduler

In [ ]:
# lr np 1e-3, 3e-4, 1e-4
# scheduler ReduceLROnPlateau

### Final Grid search

In [ ]:
# find best combo knowing the results of prev experiments

## Conclusions
During these experiments we found that the best model configuration was:
> ...

And that is the configuration we used to create our final predictions.
